# Diverse Prototypical Ensembles (DPE) — Waterbirds Reproduction

Official implementation for **Diverse Prototypical Ensembles Improve Robustness to Subpopulation Shift** (ICML 2025).

This notebook reproduces the **Waterbirds** benchmark results on Google Colab using a free T4 GPU.

## 1. Verify GPU Hardware
Colab provides a free NVIDIA GPU (usually Tesla T4 with 15 GB VRAM).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


## 2. Install Dependencies
Install the benchmark dependencies required by DPE (`netcal` and `jsonargparse`).

In [ ]:
!pip install -q netcal jsonargparse
print("Dependencies installed successfully.")


## 3. Download & Format Waterbirds Dataset
Downloads `waterbird_complete95_forest2water2.tar.gz` from Stanford NLP, unpacks images, and standardizes `metadata_waterbirds.csv`.

In [ ]:
!python scripts/prepare_waterbirds.py --data_dir data


## 4. Stage 0: ResNet-50 Feature Backbone Training & Feature Pre-extraction
Fine-tunes ImageNet-pretrained ResNet-50 for 100 epochs with ERM on Waterbirds images and pre-extracts normalized 2048-dim feature representations for validation and test splits (`feats_val.npy`, `feats_test.npy`).

*Estimated time on Colab T4:* ~25–35 minutes.

In [ ]:
!python baselines/dpe4subpop/main.py \
  --dataset_name Waterbirds \
  --data_dir data \
  --ckpt_dir checkpoints/waterbirds_stage0 \
  --model_name resnet50 \
  --pretrained_imgnet \
  --train_mode full \
  --loss_name ce \
  --epochs 100 \
  --lr 0.001 \
  --momentum 0.9 \
  --optim sgd \
  --batch_size 128 \
  --batch_size_eval 256 \
  --workers 4 \
  --force_saving_feats \
  --no_wandb \
  --seed 0


## 5. Stage 1+: Diverse Prototypical Ensemble (DPE) Training
Trains an ensemble of 16 distance-based prototype classifiers using IsoMax+ and covariance regularization on top of the frozen embeddings without subgroup labels (`train_attr no`, `subsample_type class`).

*Estimated time on Colab T4:* ~1–2 minutes.

In [ ]:
!python baselines/dpe4subpop/main.py \
  --dataset_name Waterbirds \
  --data_dir data \
  --pretrained_path checkpoints/waterbirds_stage0/ckpt_last.pt \
  --ckpt_dir checkpoints/waterbirds_dpe \
  --loss_name isomax \
  --stage 1 \
  --num_stages 16 \
  --epochs 20 \
  --batch_size 256 \
  --lr 0.001 \
  --train_attr no \
  --subsample_type class \
  --cov_reg 5.e5 \
  --entropic_scale 30 \
  --train_mode freeze \
  --ensemble_criterion wga_val \
  -ncbt \
  -sit \
  --no_wandb \
  --seed 0


## 6. Compare with Paper Reported Results

| Method | Group Info (Train / Val) | Worst-Group Accuracy | Average Accuracy |
| :--- | :---: | :---: | :---: |
| **Standard ERM** | None / None | 69.1% ± 1.8% | 97.2% ± 0.2% |
| **ERM\*** (Backbone Stage 0) | None / None | 77.9% ± 3.0% | 96.9% ± 0.3% |
| **GroupDRO** | Train & Val | 89.9% ± 0.9% | 93.5% ± 0.3% |
| **DFR** | Val only | 92.9% ± 0.4% | 94.0% ± 0.3% |
| **DPE (Stage 1+)** | **None / None** | **94.1% ± 0.2%** | **96.8% ± 0.2%** |
